<a href="https://www.kaggle.com/code/thisisrick25/imdb-sentiment-tfidf-logistic-regression?scriptVersionId=355200490" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# IMDb Sentiment Classification with TF-IDF and Logistic Regression

This notebook uses the **preprocessed IMDb reviews** to build and evaluate Logistic Regression sentiment classifiers.

Two TF-IDF representations are compared:

1. Unigrams
2. Unigrams + bigrams

The notebook covers every requested part: train/test split, model training, evaluation metrics, confusion matrix, bigram comparison, decision-threshold analysis, and inspection of five borderline test reviews.

## 1. Imports and Configuration

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 2. Load the Preprocessed IMDb Reviews

If `df` from the previous notebook is already available and contains `preprocessed_review`, it is reused.

Otherwise, the notebook automatically finds `IMDB Dataset.csv` under `/kaggle/input/` and recreates the same preprocessing pipeline.

In [ ]:
required_columns = {"preprocessed_review", "sentiment"}

if "df" in globals() and required_columns.issubset(df.columns):
    print("Using existing preprocessed IMDb DataFrame.")

else:
    dataset_path = None

    for root, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file.lower() == "imdb dataset.csv":
                dataset_path = os.path.join(root, file)
                break
        if dataset_path:
            break

    if dataset_path is None:
        raise FileNotFoundError(
            "IMDB Dataset.csv was not found under /kaggle/input/. "
            "Add the IMDb dataset to the Kaggle notebook as an Input."
        )

    print("Dataset found at:")
    print(dataset_path)

    df = pd.read_csv(dataset_path)

    import nltk
    nltk.download("punkt_tab", quiet=True)
    nltk.download("stopwords", quiet=True)
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)

    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer

    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()

    def clean_text(text):
        text = str(text).lower()
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"\b[\w.+-]+@[\w.-]+\.\w+\b", " ", text)
        text = re.sub(r"[^a-zA-Z\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def preprocess_review(text):
        tokens = word_tokenize(clean_text(text))
        tokens = [t for t in tokens if t not in stop_words]
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
        return " ".join(tokens)

    df["preprocessed_review"] = df["review"].apply(
        preprocess_review
    )

print("Dataset shape:", df.shape)

## 3. Prepare Text and Sentiment Labels

Sentiment is encoded as:

- negative = 0
- positive = 1

In [ ]:
X_text = df["preprocessed_review"].astype(str)

y = (
    df["sentiment"]
    .map({"negative": 0, "positive": 1})
    .astype(int)
)

print("Number of reviews:", len(X_text))
print("\nClass distribution:")
print(y.value_counts().sort_index())

# (a) Train-Test Split

An **80/20 stratified split** is used. Stratification preserves the class balance in both sets.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training reviews:", len(X_train_text))
print("Testing reviews:", len(X_test_text))

print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTesting class proportions:")
print(y_test.value_counts(normalize=True).sort_index())

# (b) TF-IDF Features

Two vectorizers are fitted **only on the training data**:

- unigram TF-IDF
- unigram + bigram TF-IDF

The test data is transformed using the corresponding training vocabulary.

In [ ]:
tfidf_unigram = TfidfVectorizer(
    ngram_range=(1, 1)
)

tfidf_bigram = TfidfVectorizer(
    ngram_range=(1, 2)
)

X_train_unigram = tfidf_unigram.fit_transform(X_train_text)
X_test_unigram = tfidf_unigram.transform(X_test_text)

X_train_bigram = tfidf_bigram.fit_transform(X_train_text)
X_test_bigram = tfidf_bigram.transform(X_test_text)

print("Unigram training matrix:", X_train_unigram.shape)
print("Unigram testing matrix:", X_test_unigram.shape)

print("\nUnigram + bigram training matrix:",
      X_train_bigram.shape)
print("Unigram + bigram testing matrix:",
      X_test_bigram.shape)

# (c) Logistic Regression

A separate Logistic Regression classifier is trained for each TF-IDF representation.

In [ ]:
model_unigram = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

model_bigram = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

model_unigram.fit(X_train_unigram, y_train)
model_bigram.fit(X_train_bigram, y_train)

print("Unigram model trained.")
print("Unigram + bigram model trained.")

# (d) Accuracy, Precision, Recall, and F1-Score

In [ ]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1-score": f1_score(y_test, predictions)
    }

unigram_metrics = evaluate_model(
    model_unigram,
    X_test_unigram,
    y_test
)

bigram_metrics = evaluate_model(
    model_bigram,
    X_test_bigram,
    y_test
)

results = pd.DataFrame(
    [unigram_metrics, bigram_metrics],
    index=[
        "TF-IDF Unigrams",
        "TF-IDF Unigrams + Bigrams"
    ]
)

display(results.round(4))

# (e) Confusion Matrix for the Better-Performing Model

The model with the higher F1-score is selected as the better-performing model.

In [ ]:
if bigram_metrics["F1-score"] > unigram_metrics["F1-score"]:
    best_model_name = "TF-IDF Unigrams + Bigrams"
    best_model = model_bigram
    best_X_test = X_test_bigram
else:
    best_model_name = "TF-IDF Unigrams"
    best_model = model_unigram
    best_X_test = X_test_unigram

best_predictions = best_model.predict(best_X_test)
cm = confusion_matrix(y_test, best_predictions)

print("Better-performing model:", best_model_name)
print("\nConfusion matrix:")
print(cm)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
).plot(
    ax=ax,
    values_format="d"
)

ax.set_title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

# (f) Compare the Two Models

The two models are compared using all four requested metrics. The F1-score is used as the primary criterion for deciding whether adding bigrams improves the classifier.

In [ ]:
comparison = results.copy()

print("Model comparison:")
display(comparison.round(4))

f1_difference = (
    bigram_metrics["F1-score"]
    - unigram_metrics["F1-score"]
)

accuracy_difference = (
    bigram_metrics["Accuracy"]
    - unigram_metrics["Accuracy"]
)

print(f"F1-score change from adding bigrams: {f1_difference:+.4f}")
print(f"Accuracy change from adding bigrams: {accuracy_difference:+.4f}")

if f1_difference > 0:
    print(
        "\nConclusion: Adding bigrams improved sentiment "
        "classification based on F1-score."
    )
elif f1_difference < 0:
    print(
        "\nConclusion: Adding bigrams did not improve sentiment "
        "classification based on F1-score."
    )
else:
    print(
        "\nConclusion: Adding bigrams produced the same "
        "F1-score as unigrams."
    )

# (g) Decision Threshold Analysis

Logistic Regression produces a probability for the positive class.

The following thresholds are evaluated:

- 0.3
- 0.5
- 0.7

A review is predicted positive when:

```text
P(positive) >= threshold
```

In [ ]:
positive_probabilities = best_model.predict_proba(
    best_X_test
)[:, 1]

thresholds = [0.3, 0.5, 0.7]

threshold_rows = []

for threshold in thresholds:
    predictions = (
        positive_probabilities >= threshold
    ).astype(int)

    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test, predictions
        ),
        "Recall": recall_score(
            y_test, predictions
        ),
        "F1-score": f1_score(
            y_test, predictions
        )
    })

threshold_results = pd.DataFrame(threshold_rows)

display(threshold_results.round(4))

## Precision and Recall vs Classification Threshold

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Recall"],
    marker="o",
    label="Recall"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Precision and Recall vs Classification Threshold")
plt.xticks(thresholds)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

# (i) Five Test Reviews Closest to Probability 0.5

The five most uncertain test reviews are identified by minimizing:

$$
|P(positive) - 0.5|
$$

For each review, the actual sentiment, predicted sentiment, and predicted positive probability are displayed.

In [ ]:
# Preserve the original preprocessed/test-review indices
test_positions = X_test_text.index.to_numpy()
test_reviews = X_test_text.to_numpy()
test_labels = y_test.to_numpy()

distances = np.abs(
    positive_probabilities - 0.5
)

closest_positions = np.argsort(distances)[:5]

borderline_rows = []

for position in closest_positions:
    probability = positive_probabilities[position]

    predicted = (
        "positive"
        if probability >= 0.5
        else "negative"
    )

    actual = (
        "positive"
        if test_labels[position] == 1
        else "negative"
    )

    borderline_rows.append({
        "Original Index": test_positions[position],
        "Review": test_reviews[position],
        "Actual Sentiment": actual,
        "Predicted Sentiment": predicted,
        "Predicted Probability": probability,
        "Distance from 0.5": distances[position]
    })

borderline_reviews = pd.DataFrame(borderline_rows)

display(
    borderline_reviews[
        [
            "Original Index",
            "Actual Sentiment",
            "Predicted Sentiment",
            "Predicted Probability",
            "Distance from 0.5"
        ]
    ].round(6)
)

### Full text of the five borderline reviews

In [ ]:
for number, (_, row) in enumerate(
    borderline_reviews.iterrows(),
    start=1
):
    print("=" * 100)
    print(f"Borderline Review {number}")
    print("-" * 100)
    print("Actual:", row["Actual Sentiment"])
    print("Predicted:", row["Predicted Sentiment"])
    print(
        "Probability of positive:",
        f"{row['Predicted Probability']:.6f}"
    )
    print("\nReview:")
    print(row["Review"])
    print()

# (j) Analysis of the Borderline Cases

The five reviews above are the cases where the model is most uncertain.

Look for the following linguistic patterns:

- **Ambiguity:** unclear or mixed opinion.
- **Negation:** phrases such as `not good`, `never enjoyable`, or `didn't like`.
- **Mixed sentiment:** positive and negative opinions in the same review.
- **Contrast:** `but`, `however`, `although`, `though`, etc.
- **Sarcasm or irony:** literal words do not match the intended sentiment.
- **Weak sentiment:** mostly descriptive or plot-focused language.
- **Conflicting sentiment words:** strong positive and negative terms appearing together.

The following diagnostic is only an aid; the final interpretation should be based on reading the five actual reviews.

In [ ]:
positive_words = {
    "amazing", "excellent", "great", "good", "wonderful",
    "fantastic", "brilliant", "love", "loved", "best",
    "enjoy", "enjoyed", "perfect", "awesome", "beautiful"
}

negative_words = {
    "bad", "boring", "terrible", "awful", "worst",
    "poor", "waste", "horrible", "disappointing",
    "disappointed", "hate", "hated", "dull", "failure"
}

negation_words = {
    "not", "no", "never", "neither", "nor",
    "nothing", "nowhere", "hardly", "barely"
}

contrast_words = {
    "but", "however", "although", "though",
    "despite", "except", "yet"
}

def diagnose(review):
    tokens = set(
        re.findall(r"[a-zA-Z]+", review.lower())
    )

    return {
        "Positive terms": sorted(tokens & positive_words),
        "Negative terms": sorted(tokens & negative_words),
        "Negation terms": sorted(tokens & negation_words),
        "Contrast terms": sorted(tokens & contrast_words)
    }

for number, (_, row) in enumerate(
    borderline_reviews.iterrows(),
    start=1
):
    print("=" * 100)
    print(f"Review {number} diagnostic")
    print(diagnose(row["Review"]))

# Final Discussion

## Model comparison

The results table compares Logistic Regression trained with unigram TF-IDF and unigram+bigrams TF-IDF. If the bigram model has a higher F1-score, adding bigrams improved sentiment classification. If its F1-score is lower, the additional bigram features did not improve performance.

## Threshold analysis

At a lower threshold such as 0.3, more reviews are classified as positive. This typically increases recall while potentially decreasing precision.

At 0.5, Logistic Regression uses its standard classification threshold.

At 0.7, the classifier requires stronger evidence before predicting positive. This typically increases precision while reducing recall.

The actual numerical results in the threshold table determine the observed trade-off.

## Borderline reviews

The five reviews closest to probability 0.5 represent the classifier's most uncertain decisions. They should be examined for ambiguity, negation, mixed sentiment, contrast, sarcasm, or conflicting sentiment-bearing words.

These cases also demonstrate a limitation of TF-IDF + Logistic Regression: the model uses weighted word and phrase features but does not explicitly model the full semantic relationship between words.

# Requirements Checklist

| Requirement | Status |
|---|---|
| (a) Train/test split | ✓ 80/20 stratified |
| (b) TF-IDF unigrams | ✓ |
| (b) TF-IDF unigrams + bigrams | ✓ |
| (c) Logistic Regression | ✓ |
| (d) Accuracy | ✓ |
| (d) Precision | ✓ |
| (d) Recall | ✓ |
| (d) F1-score | ✓ |
| (e) Confusion matrix for better model | ✓ |
| (f) Compare models | ✓ |
| (f) Determine whether bigrams improve classification | ✓ |
| (g) Threshold 0.3 | ✓ |
| (g) Threshold 0.5 | ✓ |
| (g) Threshold 0.7 | ✓ |
| (g) Precision/recall/F1 at each threshold | ✓ |
| Precision vs threshold plot | ✓ |
| Recall vs threshold plot | ✓ |
| (i) Five probabilities closest to 0.5 | ✓ |
| (i) Actual sentiment | ✓ |
| (i) Predicted sentiment | ✓ |
| (i) Predicted probability | ✓ |
| (j) Borderline linguistic analysis | ✓ |